# Semi-Supervised Learning

Semi-Supervised Learning uses **a small amount of labeled data + a large amount of unlabeled data**.

## Basic Workflow

**Load → EDA → Preprocessing → Separate Labeled/Unlabeled Data → Train Initial Model → Generate Pseudo-Labels / Propagate Labels → Add High-Confidence Labels → Retrain → Predict → Evaluate → Improve**

It sits between:

**Supervised Learning** → all/most data labeled

**Unsupervised Learning** → no labels

**Semi-Supervised Learning** → some labels + many unlabeled samples

## Main Idea

```text
Labeled Data
     ↓
Initial Model
     ↓
Predict Unlabeled Data
     ↓
Select High-Confidence Predictions
     ↓
Pseudo-Labeled Data
     ↓
Combine with Original Labeled Data
     ↓
Retrain
     ↓
Better Model
```

The key idea is to use the unlabeled data instead of throwing it away. Humanity occasionally discovers useful things hiding in the pile of unlabelled data.

Libraries / Tools

In [ ]:
# Common tools

import numpy as np
import pandas as pd

# scikit-learn
# → SelfTrainingClassifier
# → LabelPropagation
# → LabelSpreading

# Common supervised models can also be used
# inside self-training, for example:
# LogisticRegression
# SVC
# RandomForestClassifier

# Advanced ecosystems:
# PyTorch / TensorFlow
# → custom semi-supervised deep-learning methods

Load Dataset

In [ ]:
df = pd.read_csv("data.csv")

df.head()

EDA

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

## Create Labeled and Unlabeled Data

A semi-supervised dataset may contain labels for only part of the observations.

A common representation is:

```text
Known label → class value
Unknown label → -1
```

Example:
- `0`, `1` → labeled classes
- `-1` → unlabeled

In [ ]:
# Example target column:
# target = -1 means unlabeled

X = df.drop("target", axis=1)
y = df["target"]

print("Labeled samples:", (y != -1).sum())
print("Unlabeled samples:", (y == -1).sum())

## Preprocessing

Follow the same basic preprocessing principles as Classification:

- Handle missing values
- Remove irrelevant columns
- Separate numerical/categorical features
- Encode categorical variables
- Scale features when required
- Keep preprocessing consistent

For label propagation/spreading and distance-based methods, scaling is especially important.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols = X.select_dtypes(
    include=["object"]
).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

X_processed = preprocessor.fit_transform(X)

print("Processed shape:", X_processed.shape)

# Label Encoding for Semi-Supervised Algorithms

scikit-learn's label-propagation methods commonly use `-1` to represent an unlabeled sample.

Keep the labeled class values as their normal class labels and set unknown labels to `-1`.

# Self-Training

**Use:** Start with a supervised classifier, predict unlabeled samples, and add only sufficiently confident predictions as pseudo-labels.

Main parameter:
- `threshold` → confidence required to accept a pseudo-label

In [ ]:
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.linear_model import LogisticRegression

base_model = LogisticRegression(
    max_iter=1000
)

self_training = SelfTrainingClassifier(
    base_model,
    threshold=0.8
)

self_training.fit(
    X_processed,
    y
)

pseudo_labels = self_training.transduction_

print("Original labeled:", (y != -1).sum())
print("Pseudo-labeled:", (pseudo_labels != -1).sum())

# Label Propagation

**Use:** Labels are propagated through a graph based on similarity between samples.

Main idea:

**Similar samples → likely to have similar labels**

Important parameters:
- `kernel`
- `gamma`
- `n_neighbors`
- `max_iter`

In [ ]:
from sklearn.semi_supervised import LabelPropagation

label_prop = LabelPropagation(
    kernel="rbf",
    gamma=20,
    max_iter=1000
)

label_prop.fit(
    X_processed,
    y
)

pred_lp = label_prop.transduction_

print("Predicted labels:", np.unique(pred_lp))

# Label Spreading

**Use:** Similar to Label Propagation, but uses a graph-based spreading process with a softer regularization approach.

Common parameters:
- `kernel`
- `gamma`
- `n_neighbors`
- `alpha`

In [ ]:
from sklearn.semi_supervised import LabelSpreading

label_spread = LabelSpreading(
    kernel="rbf",
    gamma=20,
    alpha=0.2,
    max_iter=1000
)

label_spread.fit(
    X_processed,
    y
)

pred_ls = label_spread.transduction_

print("Predicted labels:", np.unique(pred_ls))

# Pseudo-Labeling

**Use:** Train a supervised model on labeled data, predict the unlabeled data, keep high-confidence predictions, then retrain using those pseudo-labels.

Concept:

```text
Labeled
   ↓
Train model
   ↓
Predict unlabeled
   ↓
Confidence threshold
   ↓
Pseudo-labels
   ↓
Retrain
```

In [ ]:
# Manual pseudo-labeling example

from sklearn.linear_model import LogisticRegression

labeled_mask = y != -1
unlabeled_mask = y == -1

X_labeled = X_processed[labeled_mask]
y_labeled = y[labeled_mask]

X_unlabeled = X_processed[unlabeled_mask]

base_model = LogisticRegression(
    max_iter=1000
)

base_model.fit(
    X_labeled,
    y_labeled
)

proba = base_model.predict_proba(
    X_unlabeled
)

pseudo_y = proba.argmax(axis=1)
confidence = proba.max(axis=1)

threshold = 0.90

high_conf = confidence >= threshold

X_pseudo = X_unlabeled[high_conf]
y_pseudo = pseudo_y[high_conf]

print("High-confidence pseudo-labels:",
      high_conf.sum())

In [ ]:
# Retrain with labeled + high-confidence pseudo-labeled data

from scipy.sparse import vstack

X_combined = vstack([
    X_labeled,
    X_pseudo
])

y_combined = np.concatenate([
    np.asarray(y_labeled),
    y_pseudo
])

final_model = LogisticRegression(
    max_iter=1000
)

final_model.fit(
    X_combined,
    y_combined
)

print("Final training samples:",
      X_combined.shape[0])

## Evaluation

If you have a **held-out labeled test set**, use normal classification metrics:

- Accuracy
- Precision
- Recall
- F1
- ROC-AUC / PR-AUC when appropriate
- Confusion Matrix

If the unlabeled data has no ground-truth labels, you cannot honestly calculate Accuracy/F1 for those samples.

Instead evaluate:
- Confidence distribution
- Number of pseudo-labels
- Stability
- Performance on a labeled validation set
- Error analysis / domain validation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_ssl(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": round(
            accuracy_score(y_true, y_pred), 3
        ),
        "Precision": round(
            precision_score(
                y_true, y_pred,
                average="weighted",
                zero_division=0
            ), 3
        ),
        "Recall": round(
            recall_score(
                y_true, y_pred,
                average="weighted",
                zero_division=0
            ), 3
        ),
        "F1": round(
            f1_score(
                y_true, y_pred,
                average="weighted",
                zero_division=0
            ), 3
        )
    }

## Model Comparison

| Method | Main idea | Best use |
|---|---|---|
| Self-Training | Add confident predictions as labels | Simple pseudo-label workflow |
| Label Propagation | Spread labels through a graph | Similar nearby samples |
| Label Spreading | Softer graph-based label spreading | Noisy/complex label relationships |
| Pseudo-Labeling | Use high-confidence model predictions | Large unlabeled dataset |

Compare methods using a **held-out labeled validation/test set**, not the unlabeled data itself.

In [ ]:
# Example comparison structure

# results = pd.DataFrame([
#     evaluate_ssl("Supervised Baseline", y_test, pred_baseline),
#     evaluate_ssl("Self-Training", y_test, pred_self_training),
#     evaluate_ssl("Label Propagation", y_test, pred_lp_test),
#     evaluate_ssl("Label Spreading", y_test, pred_ls_test)
# ])
#
# results.sort_values("F1", ascending=False)

# Model Improvement

**Baseline → Better preprocessing → Better labeled subset → Choose base classifier → Confidence threshold → Pseudo-labeling / label propagation → Retrain → Cross-validation / validation → Final model**

### Improve the labeled data
- Use the most reliable labeled examples first
- Remove incorrect labels
- Increase the labeled set when possible

### Self-Training
Tune:
- Base classifier
- Confidence `threshold`
- Number of self-training iterations

Higher threshold:
**fewer but more confident pseudo-labels**

Lower threshold:
**more pseudo-labels but higher risk of incorrect labels**

### Label Propagation / Spreading
Tune:
- `kernel`
- `gamma`
- `n_neighbors`
- `alpha` for Label Spreading

### Feature Engineering
Good representations improve similarity and pseudo-label quality.

### Class Imbalance
Check whether pseudo-labeling is creating too many examples from one class.

### Validation
Keep a labeled validation/test set untouched while generating pseudo-labels.


## Important warning

Bad pseudo-labels can reinforce the model's own mistakes.

**High-confidence wrong labels → retraining → more wrong predictions.**

So always validate the final model on data with real labels.

# Quick Revision

**Self-Training** → classifier labels unlabeled data using confidence threshold

**Pseudo-Labeling** → add confident model predictions as temporary labels

**Label Propagation** → propagate labels through sample similarity graph

**Label Spreading** → graph-based label spreading with regularization

### Main workflow

**Labeled + Unlabeled → Preprocess → Initial Model / Graph → Generate Labels → Select Reliable Labels → Retrain → Evaluate → Improve**

### Main evaluation

If labels exist:
**Accuracy / Precision / Recall / F1 / ROC-AUC / PR-AUC**

If labels do not exist:
**Confidence + stability + labeled validation performance + domain validation**

### Main improvement

**Better features → Better base model → Better confidence threshold → Better pseudo-labels → Retrain → Validate**

The central idea is simple:

**Use the small amount of labeled data to learn from the much larger amount of unlabeled data.**